# 维度变换

基本的维度变换操作函数包含了改变视图 reshape、插入新维度 expand_dims，删除维
度 squeeze、交换维度 transpose、复制数据 tile 等函数。

## 1.改变视图, tf.reshape()  

在介绍改变视图 reshape 操作之前，我们先来认识一下张量的存储(Storage)和视图
(View)的概念。张量的视图就是我们理解张量的方式，比如 shape 为[2,4,4,3]的张量𝑨，我
们从逻辑上可以理解为 2 张图片，每张图片 4 行 4 列，每个位置有 RGB 3 个通道的数据；
张量的存储体现在张量在内存上保存为一段连续的内存区域，对于同样的存储，我们可以
有不同的理解方式，比如上述张量𝑨，我们可以在不改变张量的存储下，将张量𝑨理解为 2
个样本，每个样本的特征为长度 48 的向量。同一个存储，从不同的角度观察数据，可以产
生不同的视图，这就是存储与视图的关系。视图的产生是非常灵活的，但需要保证是合
理。

In [5]:
import tensorflow as tf

# tf.reshape(),tf.reshape(x, [2, -1])  # 参数−1表示当前轴上长度需要根据张量总元素不变的法则自动推导
x = tf.range(96)
x = tf.reshape(x, [2, 4, 4, 3])  # 改变X的视图，获得4D张量，存储并未改变
# 数据仍然是 0~95 的顺序，可见数据并未改变，改变的是数据的结构
print('x:', x.shape)
print('x_ndim:', x.ndim, 'x_shape', x.shape)

x: (2, 4, 4, 3)
x_ndim: 4 x_shape (2, 4, 4, 3)


In [8]:
# 参数−1表示当前轴上长度需要根据张量总元素不变的法则自动推导
x_new = tf.reshape(x, [2, -1])
print('new_x_ndim:', x_new.ndim, 'new_x_shape:', x_new.shape)

new_x_ndim: 2 new_x_shape: (2, 48)


## 2.扩展维度，tf.expand_dims(x, axis=0)

In [ ]:
x = tf.random.uniform([28, 28], maxval=10, dtype=tf.int32)
print('x:', x.shape)
# 通过 tf.expand_dims(x, axis)可在指定的 axis 轴前可以插入一个新的维度
x = tf.expand_dims(x, axis=2)
print('expand_x:', x.shape)

x: (28, 28)
expand_x: (28, 28, 1)


同样的方法，我们可以在最前面插入一个新的维度，并命名为图片数量维度

In [13]:
# 高维度之前插入新维度
x = tf.expand_dims(x, axis=0)
print('expand_x_1:', x.shape)

expand_x_1: (1, 28, 28, 1)


## 3.删除维度,tf.squeeze(x, axis)  
删除维度 是增加维度的逆操作，与增加维度一样，删除维度只能删除长度为 1 的维
度，也不会改变张量的存储。继续考虑增加维度后 shape 为[1,28,28,1]的例子，如果希望将
图片数量维度删除，可以通过 tf.squeeze(x, axis)函数，axis 参数为待删除的维度的索引号，
例如，图片数量的维度轴 axis=0


In [14]:
x = tf.squeeze(x, axis=0)  # 不指定维度参数 axis，即 tf.squeeze(x)，那么它会默认删除所有长度为 1 的维度
print('Squeeze_x:', x.shape)

Squeeze_x: (28, 28, 1)


## 4.交换维度,tf.transpose(x, perm)  
改变视图、增删维度都不会影响张量的存储。在实现算法逻辑时，在保持维度顺序不变的条件下，仅仅改变张量的理解方式是不够的，有时需要直接调整的存储顺序，即交换维度(Transpose)。通过交换维度操作，**改变了张量的存储顺序，同时也改变了张量的视图**。

考虑图片张量 shape 为[2,32,32,3]，“图片数量、行、列、通道
数”的维度索引分别为 0、1、2、3，如果需要交换为[𝑏, 𝑐, ℎ, ]格式，则新维度的排序为
“图片数量、通道数、行、列”，对应的索引号为[0,3,1,2]，因此参数 perm 需设置为
[0,3,1,2]，实现如下

In [16]:
# “图片数量、行、列、通道 数”的维度索引分别为 0、1、2、3，如果需要交换为 [ 𝑏 , 𝑐 , ℎ , ] 格式，
# 则新维度的排序为 “图片数量、通道数、行、列”，对应的索引号为 [ 0 , 3 , 1 , 2 ]
# ，因此参数 perm 需设置为 [ 0 , 3 , 1 , 2 ]
x = tf.random.normal([2, 32, 32, 3])
x = tf.transpose(x, perm=[0, 3, 1, 2])
print('交换维度:', x.shape)

交换维度: (2, 3, 32, 32)


## 5.复制数据，tf.tile(b, [2, 1])   
可以通过 tf.tile(x, multiples)函数完成数据在指定维度上的复制操作，multiples 分别指
定了每个维度上面的复制倍数，对应位置为 1 表明不复制，为 2 表明新长度为原来长度的
2 倍，即数据复制一份，以此类推。

In [21]:
# 复制数据
b = tf.constant([1, 2])
b = tf.expand_dims(b, axis=0)  # 插入新维度，变成矩阵。
# tf.tile(x, multiples)函数完成数据在指定维度上的复制操作，multiples 分别指 定了每个维度上面的复制倍数
print('b_OLD:', b)
b = tf.tile(b, multiples=[2, 1])  # 即可在 axis=0 维度复制 1 次，在 axis=1 维度不复制
b

b_OLD: tf.Tensor([[1 2]], shape=(1, 2), dtype=int32)


<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[1, 2],
       [1, 2]], dtype=int32)>

In [23]:
# 另一个例子
x = tf.range(4)
x = tf.reshape(x, [2, 2])
x

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[0, 1],
       [2, 3]], dtype=int32)>

In [24]:
x = tf.tile(x, multiples=[1, 2])  # 列维度复制一份
x

<tf.Tensor: shape=(2, 4), dtype=int32, numpy=
array([[0, 1, 0, 1],
       [2, 3, 2, 3]], dtype=int32)>

In [25]:
x = tf.tile(x, multiples=[2, 1])  # 行维度复制一份
x

<tf.Tensor: shape=(4, 4), dtype=int32, numpy=
array([[0, 1, 0, 1],
       [2, 3, 2, 3],
       [0, 1, 0, 1],
       [2, 3, 2, 3]], dtype=int32)>